# demo

> Interactive demos on the `learning` shelf: store them, make them in the background after repeated misses, and log each prediction.

A demo is one interactive figure from [rupaka](https://github.com/vedicreader/rupaka). It asks the learner for a prediction first. It shows the result after the learner locks in the prediction.
This module keeps demos on the `learning` shelf. Concepts and attempts are on the same shelf. The module decides when the tutor makes a demo. It runs that job in the background, so the learner never waits.

| rule | what happens |
|---|---|
| a miss on a concept that has a demo | the result of the question gets the demo, and the session shows it under the lesson |
| two misses with a saved misconception, or a concept with `visual: true` in its meta | the tutor queues a job: a model writes a demo, and rupaka checks it |
| the learner asks ("Show me a demo" on the concept page) | the same job starts immediately |
| a walk opens a step, and the next step has a demo or an order check | the job for the next step starts, so its demo is ready in time |
| a prediction in a demo | an attempt with `kind='demo'`, and an answer to the learner model: depth 2, or depth 3 with a "why" line |

In [ ]:
#| default_exp demo

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # and the settings, with the saved prompts, go to a temporary folder

Import the modules that this notebook uses.

In [ ]:
#| export
import json, time
from concurrent.futures import ThreadPoolExecutor
from fastcore.utils import *
import rupaka
from rupaka.spec import upgrade, gallery
from rupaka.agent import author
from abhyasa.store import Learning, slugify, _src
from abhyasa.tutor import CLOSE_HOOKS
from abhyasa.model import update
from abhyasa.method import way
from abhyasa.settings import val


## Storage

A demo is one vault document. Its `source` is `learning://demo/<slug>`, its `kind` is `demo`, and its meta holds the spec.

| meta | value |
|---|---|
| `concept` | the slug of the concept, the link |
| `spec` | the rupaka spec |
| `schema`, `rupaka` | the schema of the spec, and the rupaka version that checked it |
| `by` | `model:<name>` or `hand` |
| `checked` | the time of the last passing check, or None |
| `aim` | `''` for a demo, or `'order'` for an order check: the learner puts steps in order |
| `stats` | `shown`: predictions made; `right`: predictions that were right |

The text of the document is the title, the question, the gate prompts and the captions, so hybrid search finds a demo.

In [ ]:
#| export
def demo_text(spec) -> str:
    "The searchable text of a demo: title, ask, gate prompts and captions."
    def walk(bs):
        for b in bs or []:
            yield from (b[k] for k in ('prompt', 'text') if isinstance(b.get(k), str))
            yield from walk(b.get('blocks'))
    return '\n\n'.join(t for t in (spec.get('title'), spec.get('ask'), *walk(spec.get('blocks'))) if t)

`put_demo` adds a demo, or replaces the demo with the same slug. It keeps the stats of the old demo. The vault keeps only one document for each text. So the text ends with the slug and the concept, and two demos with the same spec stay two documents. A model picks the id of a spec. When another concept, or another aim, already uses that id, `put_demo` puts the slug of the concept (and the aim) in front of the id.

In [ ]:
#| export
@patch
def put_demo(self:Learning, spec:dict, concept:str, by:str='hand', checked:float=None, aim:str='') -> AttrDict:
    "Store `spec` as a demo of `concept` with the aim `aim`."
    spec = upgrade(spec)
    slug = slugify(spec['id'])
    old = self.demo(slug) or {}
    if old and (old.get('concept'), old.get('aim', '')) != (concept, aim):   # a model picks the id
        slug = slugify('-'.join(x for x in (concept, aim, spec['id']) if x)); old = self.demo(slug) or {}
    m = dict(slug=slug, concept=concept, spec=spec, schema=spec['rupaka'], rupaka=rupaka.__version__, by=by,
             checked=checked, aim=aim, at=time.time(), stats=old.get('stats') or dict(shown=0, right=0))
    r = self.v.add(f'{demo_text(spec)}\n\nDemo {slug} for {concept}', title=spec.get('title') or slug, source=_src('demo', slug), kind='demo', meta=m)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)
    return self.demo(slug)

`demo` gets one demo by slug. `demos` lists the demos of a concept, newest first. `demo_for` gives the newest one. With an `aim`, `demo_for` gives the newest demo with that aim.

In [ ]:
#| export
def _meta(r): return AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']})

@patch
def demo(self:Learning, slug:str) -> AttrDict|None:
    "One demo by slug."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('demo', slug)]))
    return _meta(r) if r else None

@patch
def demos(self:Learning, concept:str=None) -> L:
    "The demos of `concept` (or all demos), newest first."
    rs = self._docs('demo', *(("json_extract(meta, '$.concept')=?", [concept]) if concept else ()))
    return L(sorted(map(_meta, rs), key=lambda d: -d.get('at', 0)))

@patch
def demo_for(self:Learning, concept:str, aim:str=None) -> AttrDict|None:
    "The newest demo of `concept`, or of `concept` with the aim `aim`; None when there is none."
    return first(d for d in self.demos(concept) if aim is None or d.get('aim', '') == aim)

### Tests for storage

Make a shelf in a temporary vault, with one concept and the sandhi demo from the rupaka gallery.

In [ ]:
from abhyasa.store import temp_learning
lr = temp_learning()
lr.put('Vowel sandhi', 'Two vowels at a word boundary join by rule.', 'sanskrit', slug='sa-sandhi-vowels')
spec = gallery()['vowel-sandhi']
d = lr.put_demo(spec, 'sa-sandhi-vowels')
test_eq((d.slug, d.concept, d.by, d.stats), ('vowel-sandhi', 'sa-sandhi-vowels', 'hand', dict(shown=0, right=0)))

`demo_for` finds the demo by its concept. `demos` gives an empty list for an unknown concept. A search for the words of the caption finds the demo.

In [ ]:
test_eq(lr.demo_for('sa-sandhi-vowels').spec, spec)
test_eq(lr.demos('nope'), [])
test_eq(lr.search('rule stops another starts', kind='demo')[0]['doc_id'], d.doc_id)

The same id for another concept gets a new slug, so it does not replace the first demo.

In [ ]:
lr.put('Guna', 'a + i gives e.', 'sanskrit', slug='guna')
test_eq(lr.put_demo(spec, 'guna').slug, 'guna-vowel-sandhi')
test_eq(lr.demo('vowel-sandhi').concept, 'sa-sandhi-vowels')
lr.drop('guna')

An order demo with the same id and the same concept also gets a new slug. `demo_for` with an aim finds only the demo with that aim. With no aim, it finds the newest demo.

In [ ]:
o = lr.put_demo(spec, 'sa-sandhi-vowels', aim='order')
test_eq((o.slug, o.aim), ('sa-sandhi-vowels-order-vowel-sandhi', 'order'))
test_eq((lr.demo_for('sa-sandhi-vowels', aim='').slug, lr.demo_for('sa-sandhi-vowels', aim='order').slug), ('vowel-sandhi', o.slug))
test_eq(lr.demo_for('sa-sandhi-vowels').slug, o.slug)
lr.v.forget(o.doc_id)

A demo is not a concept, so the list of concepts stays the same.

In [ ]:
test_eq(lr.concepts().attrgot('slug'), ['sa-sandhi-vowels'])

## Predictions

The demo page sends each prediction. `record_prediction` stores it as an attempt with `kind='demo'`. The attempt has the gate, the pick, the right answer and the "why" line.
It counts the prediction in the stats of the demo.
A prediction also shows what the learner understands. So it goes to the learner model as an answer. The depth is 2 (recall), or 3 (explain) when the learner wrote why.
The rules are the same as for a question. A right pick with high confidence is solid. A wrong pick with high confidence is a misconception.

In [ ]:
#| export
def _gate(spec, gid):
    def walk(bs):
        for b in bs or []:
            if b.get('type') == 'gate' and b.get('id') == gid: return b
            if (g := walk(b.get('blocks'))): return g
    return walk(spec.get('blocks')) or {}

_OWN = {'concept', 'question', 'answer', 'thinking', 'correct', 'confidence', 'depth', 'kind', 'session', 'final', 'method', 'feedback', 'demo', 'gate', 'expected', 'outcome', 'hints', 'dont_know', 'grade', 'first_error', 'misconception', 'id', 'at'}

@patch
def record_prediction(self:Learning, slug:str, gate:str, picked, ans, ok:bool, why:str='', context:str='', session:str='', confidence:int=None,
                      kind:str='demo', method:str='demo', **meta) -> AttrDict:
    "File one prediction in demo `slug` as an attempt, count it for `method`, and update the learner model of its concept. `meta` goes on the attempt."
    if (bad := _OWN & meta.keys()): raise ValueError(f'meta cannot use {sorted(bad)}')
    d = self.demo(slug)
    if d is None: raise KeyError(f'no demo {slug!r}')
    st = d.get('stats') or {}
    self.v.set_meta(d.doc_id, stats=dict(shown=st.get('shown', 0) + 1, right=st.get('right', 0) + bool(ok)))
    q = f"{context} {_gate(d.spec, gate).get('prompt', gate)}".strip()
    depth = 3 if why.strip() else 2
    if (c := self.concept(d.concept)) is not None:
        u = update(c, depth, bool(ok), confidence)
        out = u.pop('kind'); self.update(d.concept, **u)
        self.credit_method(method, c.get('subject', ''), out)   # a demo is a way to learn
    else: out = ''
    return self.record(d.concept, q, str(picked), why, bool(ok), confidence, depth, kind=kind, session=session, final=True, method=method,
                       feedback=f'The answer is {ans}.', demo=slug, gate=gate, expected=str(ans), outcome=out, **meta)

Record a wrong prediction with a "why" line. Check the attempt fields, the question text and the stats of the demo.

In [ ]:
c0 = lr.concept('sa-sandhi-vowels')
a = lr.record_prediction('vowel-sandhi', 'join', 'ai', 'e', False, why='a and i make a diphthong', context='For a + i:', session='s1')
test_eq((a.kind, a.correct, a.thinking, a.demo, a.expected), ('demo', False, 'a and i make a diphthong', 'vowel-sandhi', 'e'))
test_eq(a.question, 'For a + i: What do the two vowels become?')
test_eq(lr.demo('vowel-sandhi').stats, dict(shown=1, right=0))

A prediction credits the way `demo`, and keeps the meta of the walk. A step in an order check uses `kind='order'`.

In [ ]:
lrp = temp_learning(); lrp.put('Reciprocal rank fusion', 'Sum 1/(k + rank).', 'ml')
dp = lrp.put_demo(gallery()['reciprocal-rank-fusion'], 'reciprocal-rank-fusion')
gp = first(b['id'] for b in dp.spec['blocks'] if b.get('type') == 'gate')
ap = lrp.record_prediction(dp.slug, gp, 1, 1, True, session='s1', walk='w1', step=2)
assert (ap.kind, ap.method, ap.walk, ap.step) == ('demo', 'demo', 'w1', 2)
assert 'demo' in lrp.stats().methods['ml'] and 'diagram' not in lrp.stats().methods['ml']
assert lrp.record_prediction(dp.slug, gp, 0, 1, False, kind='order', method='order').kind == 'order'
test_fail(lambda: lrp.record_prediction(dp.slug, gp, 1, 1, True, depth=5), contains='meta')   # a key of the attempt cannot be meta

The learner model of the concept gets the prediction. A wrong pick with a "why" line is an answer at depth 3. So the ceiling falls to 3, and the concept is seen one more time.

In [ ]:
c1 = lr.concept('sa-sandhi-vowels')
test_eq((c1.ceiling, c1.seen), (3, c0.seen + 1))
test_eq(a.depth, 3)

## When to make a demo

`want_demo` says if a concept should get a demo. It is true after `checks.demo_misses` misses (2 by default) with a saved misconception, or when the concept meta has `visual: true`.

In [ ]:
#| export
def want_demo(c) -> bool:
    "True when concept `c` should get a demo."
    return bool(c and (c.get('visual') or (c.get('misses', 0) >= val('checks.demo_misses') and c.get('misconceptions'))))

Check the rule. One miss, or misses with no misconception, give no demo. Two misses with a misconception, or `visual`, give a demo.

In [ ]:
assert not want_demo(dict(misses=1, misconceptions=['x'])) and not want_demo(dict(misses=3, misconceptions=[]))
assert want_demo(dict(misses=2, misconceptions=['x'])) and want_demo(dict(visual=True))

## Make a demo

`make_demo` asks the model for a demo with `rupaka.author`, and stores the demo. The loop validates the spec. It checks the spec in headless Chrome when a browser is available. It repairs the spec up to two times.
When the model cannot make a good demo, `make_demo` returns None and stores nothing. The text lesson stays.

With `aim='order'`, the demo is an order check. The prompt asks for one gate and one ranklist of 3 to 6 steps, and the examples are the gallery demos that have a ranklist.

In [ ]:
#| export
ORDER_AIM = 'Make this demo an order check. Use one gate and one ranklist of 3 to 6 steps of the idea.'

def _types(bs):
    "The block types in the blocks `bs`, with the blocks inside them."
    for b in bs or []: yield b.get('type'); yield from _types(b.get('blocks'))

def make_demo(learning:Learning, ask, slug:str, first_error:str='', check=None, log:list=None, aim:str='') -> AttrDict|None:
    "A new checked demo for concept `slug` with the aim `aim`, stored on the shelf; None when the model fails."
    c = learning.concept(slug)
    if c is None: return None
    if check is None:
        from rupaka.check import can_check
        check = can_check()
    ex, ask_ = None, ask
    if aim == 'order':
        ex = [s for s in gallery().values() if 'ranklist' in _types(s.get('blocks'))]
        ask_ = lambda p, sp='': ask(f"{p}\n\n{way('order')} {ORDER_AIM}", sp)
    spec = author(c, ask_, first_error, examples=ex, check=check, log=log)
    if spec is None: return None
    return learning.put_demo(spec, slug, by=f"model:{getattr(ask, 'model', 'unknown')}", checked=time.time() if check else None, aim=aim)

A fake model acts as the agent. It replies with a demo for the concept, or with text that is not JSON.

In [ ]:
from abhyasa.llm import Fake
good = {**spec, 'id': 'sandhi-by-model'}
fk = Fake({'You design one interactive demo': good})
d = make_demo(lr, fk, 'sa-sandhi-vowels', check=False)
test_eq((d.slug, d.by, d.checked), ('sandhi-by-model', 'model:unknown', None))
test_eq(lr.demo_for('sa-sandhi-vowels').slug, 'sandhi-by-model')

When the model never gives a good spec, `make_demo` stores nothing. The log names the errors.

In [ ]:
log = []
test_eq(make_demo(lr, Fake({'You design one interactive demo': 'no demo today'}), 'sa-sandhi-vowels', check=False, log=log), None)
test_eq(len(log), 3)
test_eq(len(lr.demos('sa-sandhi-vowels')), 2)

An order demo: the prompt asks for a ranklist and shows the ranklist example from the gallery. The demo keeps its aim.

In [ ]:
fo = Fake({'You design one interactive demo': {**spec, 'id': 'sandhi-order'}})
d = make_demo(lr, fo, 'sa-sandhi-vowels', check=False, aim='order')
p = fo.calls[0][0]
assert ORDER_AIM in p and 'Reciprocal rank fusion' in p and 'Vowel sandhi' not in p.split('Examples')[1]
test_eq((d.slug, d.aim), ('sandhi-order', 'order'))
lr.v.forget(d.doc_id)

## The background job

`queue_demo` runs `make_demo` in the thread pool of the tutor. It runs only one job at a time for each concept and aim. A second call while the job runs gives the same job.
The key of a job is the concept slug, or `slug:aim` for an aim. `job_state` tells you if the job runs, failed or is done. `job_age` gives the seconds since the job started, so a page can stop the wait.

In [ ]:
#| export
_POOL = ThreadPoolExecutor(1)

def _key(slug, aim): return f'{slug}:{aim}' if aim else slug

def queue_demo(tutor, slug:str, first_error:str='', aim:str=''):
    "Start a background job that makes a demo for `slug` with the aim `aim`; the running job if there is one."
    k = _key(slug, aim)
    j = tutor.jobs.get(k)
    if j is not None and not j.done(): return j
    j = tutor.jobs[k] = (tutor.pool or _POOL).submit(make_demo, tutor.learning, tutor.ask, slug, first_error, aim=aim)
    j.at = time.time()
    return j

def job_state(tutor, slug:str, aim:str='') -> str:
    "'running', 'failed', 'done' or 'none' for the demo job of `slug` with the aim `aim`."
    j = tutor.jobs.get(_key(slug, aim))
    if j is None: return 'none'
    if not j.done(): return 'running'
    return 'done' if not j.exception() and j.result() else 'failed'

def job_age(tutor, slug:str, aim:str='') -> float:
    "The seconds since the demo job of `slug` with the aim `aim` started; 0 when there is no job."
    j = tutor.jobs.get(_key(slug, aim))
    return time.time() - getattr(j, 'at', time.time()) if j is not None else 0.

## The tutor hook

`on_close` runs after each question closes (it is in `CLOSE_HOOKS`). After a miss, it attaches the newest plain demo of the concept to the result. An order check is not a demo for a lesson, so `on_close` does not use it.
When there is no demo yet and the concept wants one, it queues the job. Then it marks the result with `demo_pending`. A pass adds nothing.

In [ ]:
#| export
def on_close(tutor, q, r):
    "After a miss: attach the newest demo of the concept to the result, or queue a job when the concept wants a demo."
    if r.correct: return r
    if (d := tutor.learning.demo_for(q.concept, aim='')): r.demo = d.slug   # a plain demo, not an order check
    elif want_demo(tutor.learning.concept(q.concept)): queue_demo(tutor, q.concept, r.get('first_error', '')); r.demo_pending = True
    return r

CLOSE_HOOKS[:] = [f for f in CLOSE_HOOKS if getattr(f, '__name__', '') != 'on_close'] + [on_close]

### Tests for the hook

Make a session where each answer is a miss with high confidence and the same misconception. The fake model writes the questions, grades, lessons and demos.

In [ ]:
from abhyasa.tutor import Tutor
from concurrent.futures import ThreadPoolExecutor
def grader(p, sp): return dict(correct=False, grade=1, first_error='You add the vowels.', misconception='vowels just add', feedback='Look at the rule.')
fk = Fake({'You grade one answer': grader, 'Teach the smallest step': 'One step. Check: what is a + u?',
           'You design one interactive demo': good}, dflt=dict(question='What is a + i?', answer='e', rubric=['e']))
lr2 = temp_learning()
lr2.put('Vowel sandhi', 'Two vowels join by rule.', 'sanskrit', slug='sa-sandhi-vowels')
t = Tutor(lr2, fk, context=False, pool=ThreadPoolExecutor(1))
def miss():
    q = t.next(focus='sa-sandhi-vowels')
    return t.give_up(q.id, confidence=90)

The first miss queues no job, because there is only one miss.

In [ ]:
r1 = miss()
test_eq((r1.get('demo'), r1.get('demo_pending'), lr2.concept('sa-sandhi-vowels').misses), (None, None, 1))

A miss with a misconception saves the misconception. The second such miss queues one job. A third call while the job runs does not queue another job.

In [ ]:
lr2.update('sa-sandhi-vowels', misconceptions=['vowels just add'])
r2 = miss()
j = t.jobs['sa-sandhi-vowels']
assert r2.demo_pending and (queue_demo(t, 'sa-sandhi-vowels') is j or j.done())
t.jobs['sa-sandhi-vowels'].result()
test_eq(job_state(t, 'sa-sandhi-vowels'), 'done')

Now the concept has a demo, so the next miss gets its slug in the result.

In [ ]:
r3 = miss()
test_eq(r3.demo, 'sandhi-by-model')

A pass gets no demo.

In [ ]:
q = t.next(focus='sa-sandhi-vowels')
r = t._close(t.open[q.id], True, 80, AttrDict(correct=True, grade=9))
assert 'demo' not in r

A job with an aim has its own key. So an order job does not block the demo job of the same concept. A new job has an age near 0.

In [ ]:
jo = queue_demo(t, 'sa-sandhi-vowels', aim='order')
assert t.jobs['sa-sandhi-vowels:order'] is jo and job_age(t, 'sa-sandhi-vowels', 'order') < 5
jo.result()
test_eq((job_state(t, 'sa-sandhi-vowels', 'order'), lr2.demo_for('sa-sandhi-vowels', aim='order').aim), ('done', 'order'))
test_eq((job_state(t, 'nope'), job_age(t, 'nope')), ('none', 0.))

A concept with only an order demo gets no demo after a miss. The order demo is for a walk step.

In [ ]:
lr2.put('Guna', 'a + i gives e.', 'sanskrit', slug='guna')
lr2.put_demo({**spec, 'id': 'guna-order'}, 'guna', aim='order')
assert 'demo' not in on_close(t, AttrDict(concept='guna'), AttrDict(correct=False))

## The gallery

`import_gallery` stores the rupaka gallery demos for the concepts that they teach. A demo goes to a concept when the concept slug, or the slugified concept name, matches the concept or the title of the demo.
For other cases, `links` maps a gallery id to a concept slug.

In [ ]:
#| export
def import_gallery(learning:Learning, links:dict=None) -> L:
    "Store the rupaka gallery demos for the matching concepts; return the stored demos."
    links, out = links or {}, L()
    cs = learning.concepts()
    for gid, s in gallery().items():
        keys = {s.get('concept'), slugify(s.get('title', '')), gid}
        slug = links.get(gid) or first(c.slug for c in cs if c.slug in keys or slugify(c.name) in keys)
        if slug: out.append(learning.put_demo(s, slug))
    return out

The gallery demo goes to its matching concept. `links` sends one more gallery demo to the concept that you name.

In [ ]:
test_eq(import_gallery(lr2).attrgot('slug'), ['vowel-sandhi'])
test_eq(import_gallery(lr2, links={'nakshatra-pada': 'sa-sandhi-vowels'}).attrgot('slug'), ['nakshatra-pada', 'vowel-sandhi'])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()